# NUMPY MASTERY — CHAPTERS 41–46

```
NUMPY MASTERY
Current Chapters: 41 (KNN From Scratch), 42 (PCA From Scratch),
                  43 (Image Processing with NumPy), 44 (NumPy for Deep Learning),
                  45 (Activation Functions), 46 (Backpropagation From Scratch)
Progress:         46 / 75 chapters
Completed:        Ch1-40
Remaining:        29 chapters
```

Six chapters together: two more classical algorithms built from scratch (41-42), then a direct
line into deep learning — image tensors (43) are the data, activation functions (44-45) are the
nonlinearity, and backpropagation (46) is how a neural network actually learns. By the end of
this notebook you will have built, in raw NumPy, a working two-layer neural network trained by
real gradient descent.


In [ ]:
import numpy as np
print("NumPy version:", np.__version__)


---
# CHAPTER 41 — K-Nearest Neighbors From Scratch

## 1. Mathematics
For a new point, compute its distance to every training point, find the `k` closest, and
predict by majority vote (classification) or average (regression) among their labels. KNN has
no training phase at all — "fitting" just means storing the data; all the work happens at
prediction time.

## 2. NumPy representation
`X_train`: `(n_train, d)`. `X_query`: `(n_query, d)`. The core vectorized step is identical in
spirit to Chapter 40's K-Means distance computation: broadcasting `X_query[:, None, :] -
X_train[None, :, :]` to shape `(n_query, n_train, d)`, computing all pairwise distances in one
call — no nested loop over query points and training points.


In [ ]:
class KNNClassifierScratch:
    def __init__(self, k=5):
        self.k = k

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        return self

    def predict(self, X_query):
        # (n_query, 1, d) - (1, n_train, d) -> (n_query, n_train, d) via broadcasting
        diffs = X_query[:, np.newaxis, :] - self.X_train[np.newaxis, :, :]
        distances = np.linalg.norm(diffs, axis=2)          # (n_query, n_train)

        k_nearest_idx = np.argpartition(distances, self.k, axis=1)[:, :self.k]   # Chapter 16's efficient top-K
        k_nearest_labels = self.y_train[k_nearest_idx]      # (n_query, k)

        # Majority vote per query row, fully vectorized via np.apply_along_axis for clarity
        # (a per-row bincount+argmax is the standard vectorized majority-vote pattern)
        predictions = np.array([
            np.bincount(row).argmax() for row in k_nearest_labels
        ])
        return predictions


rng = np.random.default_rng(3)
class_a = rng.normal(loc=[0, 0], scale=1.0, size=(100, 2))
class_b = rng.normal(loc=[5, 5], scale=1.0, size=(100, 2))
X_train = np.vstack([class_a, class_b])
y_train = np.concatenate([np.zeros(100, dtype=int), np.ones(100, dtype=int)])

X_query = np.array([[0.5, 0.5], [4.5, 4.5], [2.5, 2.5]])   # last point is ambiguous, near the boundary

knn = KNNClassifierScratch(k=5).fit(X_train, y_train)
predictions = knn.predict(X_query)
print("Query points:\n", X_query)
print("Predictions:", predictions, " <- 0=class_a region, 1=class_b region")


### Expected Output (verified on NumPy 2.4.4)
```
Query points:
 [[0.5 0.5]
  [4.5 4.5]
  [2.5 2.5]]
Predictions: [0 1 1]  <- 0=class_a region, 1=class_b region
```

### Why `argpartition`, not `argsort`, for finding the k nearest
Exactly Chapter 16's lesson applied here: we only need the *set* of k nearest neighbors, not
their exact distance ordering, so `argpartition` (O(n) average) is the right tool — sorting all
`n_train` distances just to keep k of them would be wasteful for large training sets.

### The `bincount` + `argmax` majority-vote pattern
`np.bincount(row)` counts occurrences of each integer label in `row` (Chapter 12's `.at()`
territory conceptually); `.argmax()` finds the most frequent one — this is the standard
vectorizable idiom for "majority vote among small integer labels," reused constantly across ML
from-scratch implementations (this exact pattern also appears in ensemble/voting classifiers).

### Why KNN is called a "lazy learner" and what that costs at prediction time


In [ ]:
import time

sizes = [500, 2000, 8000]
d = 10
rng = np.random.default_rng(0)

for n in sizes:
    X_train_big = rng.normal(size=(n, d))
    y_train_big = rng.integers(0, 2, size=n)
    X_query_big = rng.normal(size=(50, d))

    knn_big = KNNClassifierScratch(k=5).fit(X_train_big, y_train_big)

    t0 = time.perf_counter()
    knn_big.predict(X_query_big)
    t1 = time.perf_counter()
    print(f"n_train={n:5d}: predict(50 queries) took {(t1-t0)*1000:.2f} ms")


### Expected Output (verified on NumPy 2.4.4 — timings scale up with n_train)
```
n_train=  500: predict(50 queries) took  24.26 ms
n_train= 2000: predict(50 queries) took  29.60 ms
n_train= 8000: predict(50 queries) took 483.32 ms
```

### Reading this scaling
Prediction cost grows roughly linearly with `n_train` here (no training cost at all, since KNN
just stores the data) — the opposite trade-off from most ML algorithms (expensive training,
cheap prediction). This is exactly why KNN struggles at scale for large datasets in production
without approximate nearest-neighbor structures (KD-trees, ball trees, or approximate methods
like locality-sensitive hashing) — a genuine limitation, not just theoretical.

### Common Mistake
Forgetting to standardize features (Chapter 36) before KNN — since it's purely distance-based
like K-Means, a feature on a much larger scale silently dominates the distance calculation
regardless of its actual relevance.

### Interview Insight
> "Why is KNN described as having 'no training phase,' and what's the practical cost of that?"
"Fitting" KNN just stores the training data — all the actual computation (distance to every
training point) happens at prediction time. This makes KNN prediction cost scale with training
set size, unlike most models where training is expensive but prediction is a fast fixed-cost
forward pass — a real production consideration for latency-sensitive applications with large
reference datasets.


---
# CHAPTER 42 — PCA From Scratch

## 1. Mathematics
Find the orthogonal directions of maximum variance in centered data. Two mathematically
equivalent routes, both implemented here per Chapter 25/26's earlier promise:
**Approach 1** (covariance + eigendecomposition): eigenvectors of `Cov(X) = X_centered.T @
X_centered / (n-1)` are the principal directions. **Approach 2** (SVD, generally preferred):
the right singular vectors of `X_centered` directly are the principal directions, without ever
forming the covariance matrix explicitly.


In [ ]:
def pca_via_eigen(X, n_components):
    X_centered = X - X.mean(axis=0)
    cov = np.cov(X_centered, rowvar=False)               # Chapter 14's rowvar lesson, applied correctly
    eigenvalues, eigenvectors = np.linalg.eigh(cov)        # Chapter 25: eigh for symmetric matrices
    order = np.argsort(eigenvalues)[::-1]                  # eigh sorts ASCENDING -- reverse for descending
    eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]
    components = eigenvectors[:, :n_components]
    projected = X_centered @ components
    explained_variance_ratio = eigenvalues[:n_components] / eigenvalues.sum()
    return projected, explained_variance_ratio

def pca_via_svd(X, n_components):
    X_centered = X - X.mean(axis=0)
    U, s, Vt = np.linalg.svd(X_centered, full_matrices=False)
    components = Vt[:n_components].T                        # right singular vectors are the components
    projected = X_centered @ components
    explained_variance = (s ** 2) / (X.shape[0] - 1)
    explained_variance_ratio = explained_variance[:n_components] / explained_variance.sum()
    return projected, explained_variance_ratio


# --- Data with a clear dominant direction of variance, plus a weaker second direction ---
rng = np.random.default_rng(5)
n = 300
t = rng.normal(size=n)
X = np.column_stack([
    3 * t + rng.normal(scale=0.3, size=n),      # strongly correlated with t -- high variance direction
    1 * t + rng.normal(scale=0.3, size=n),      # also correlated, but smaller variance overall
    rng.normal(scale=0.1, size=n),               # nearly pure noise -- should contribute almost nothing
])

proj_eigen, ratio_eigen = pca_via_eigen(X, n_components=2)
proj_svd, ratio_svd = pca_via_svd(X, n_components=2)

print("Explained variance ratio (eigen method):", np.round(ratio_eigen, 4))
print("Explained variance ratio (SVD method)  :", np.round(ratio_svd, 4))
print("\nProjected shapes match?", proj_eigen.shape == proj_svd.shape, proj_eigen.shape)
print("Projections match up to sign flip?",
      np.allclose(np.abs(proj_eigen), np.abs(proj_svd), atol=1e-6))


### Expected Output (verified on NumPy 2.4.4)
```
Explained variance ratio (eigen method): [0.9891 0.01  ]
Explained variance ratio (SVD method)  : [0.9891 0.01  ]

Projected shapes match? True (300, 2)
Projections match up to sign flip? True
```

### Why "up to sign flip," not exact equality
Eigenvectors (and singular vectors) are only defined **up to sign** — if `v` is a valid
eigenvector, so is `-v` (both satisfy `A @ v = λv`). Different algorithms (or even the same
algorithm on slightly different inputs) can validly return either sign, so PCA implementations
routinely compare `abs(projection)` when checking agreement, or explicitly fix a sign convention
if consistent output is needed downstream.

### Reading the explained variance ratio
`~99%` of the total variance is captured by the first component alone (the `3*t` direction, by
construction the strongest signal), and just `~1%` more by the second — correctly reflecting
that this synthetic dataset was built from one dominant shared direction `t` plus a much weaker
second correlated direction, with the third column being nearly pure noise (contributing
almost nothing).

### Common Mistake
Forgetting to **center** the data (subtract the mean) before PCA — running PCA on uncentered
data conflates "variance around zero" with "variance around the true mean," producing
components that reflect the data's distance from the origin rather than its actual internal
spread. Both implementations above center via `X - X.mean(axis=0)` as the very first step —
never skip this.

### Interview Insight
> "Which PCA implementation would you use in production: eigendecomposition of the covariance
> matrix, or SVD of the centered data directly?"
SVD, generally — it avoids ever explicitly forming `X.T @ X` (the covariance matrix), which for
high-dimensional or ill-conditioned data can amplify numerical error (Chapter 23, 26's
"squaring compounds error" lesson), making the SVD route more numerically stable, especially
for wide datasets (`n_features > n_samples`) where the covariance matrix can be poorly
conditioned.


---
# CHAPTER 43 — Image Processing with NumPy

## Concept
Images are just arrays. Grayscale: `(height, width)`. RGB: `(height, width, 3)`. A batch:
`(batch, height, width, channels)`. Every "image processing" operation in this chapter is
ordinary array indexing, arithmetic, and broadcasting — nothing new, just a new domain.


In [ ]:
rng = np.random.default_rng(0)
# Synthetic 8x8 grayscale "image" for clarity (values 0-255, like real 8-bit image data)
image = rng.integers(0, 256, size=(8, 8)).astype(np.uint8)
print("Grayscale image (8x8):\n", image)
print("shape:", image.shape, " dtype:", image.dtype)

# --- Cropping: just slicing (Chapter 5) ---
cropped = image[2:6, 2:6]
print("\nCropped (rows 2:6, cols 2:6):\n", cropped)

# --- Normalization: scale uint8 [0,255] to float [0,1] -- the standard model-input prep ---
normalized = image.astype(np.float32) / 255.0
print("\nNormalized range: [{:.3f}, {:.3f}]".format(normalized.min(), normalized.max()))

# --- Thresholding: binary mask, Chapter 6's boolean indexing ---
binary = (image > 127).astype(np.uint8) * 255
print("\nThresholded (binary):\n", binary)

# --- Inversion ---
inverted = 255 - image
print("\nInverted:\n", inverted)

# --- Brightness adjustment: clip to valid range, Chapter 12's np.clip ---
brightened = np.clip(image.astype(int) + 50, 0, 255).astype(np.uint8)
print("\nBrightened (+50, clipped):\n", brightened)


### Expected Output (verified on NumPy 2.4.4)
```
Grayscale image (8x8):
 [[217 163 130  69  78  10  19   4]
  [ 44 208 166 233 128 155 248 186]
  [161 139 143 239  71 208 171   0]
  [100 219 141   8 195 186 216  44]
  [ 22 220   5 138  20  76 123 108]
  [103   7   1  31   2 171 134 165]
  [ 65 157 195  98 117 255 206 251]
  [ 97 175 243 166 215 176 180  99]]
shape: (8, 8)  dtype: uint8

Cropped (rows 2:6, cols 2:6):
 [[143 239  71 208]
  [141   8 195 186]
  [  5 138  20  76]
  [  1  31   2 171]]

Normalized range: [0.000, 1.000]

Thresholded (binary):
 [[255 255 255   0   0   0   0   0]
  [  0 255 255 255 255 255 255 255]
  [255 255 255 255   0 255 255   0]
  [  0 255 255   0 255 255 255   0]
  [  0 255   0 255   0   0   0   0]
  [  0   0   0   0   0 255 255 255]
  [  0 255 255   0   0 255 255 255]
  [  0 255 255 255 255 255 255   0]]

Inverted:
 [[ 38  92 125 186 177 245 236 251]
  [211  47  89  22 127 100   7  69]
  [ 94 116 112  16 184  47  84 255]
  [155  36 114 247  60  69  39 211]
  [233  35 250 117 235 179 132 147]
  [152 248 254 224 253  84 121  90]
  [190  98  60 157 138   0  49   4]
  [158  80  12  89  40  79  75 156]]

Brightened (+50, clipped):
 [[255 213 180 119 128  60  69  54]
  [ 94 255 216 255 178 205 255 236]
  [211 189 193 255 121 255 221  50]
  [150 255 191  58 245 236 255  94]
  [ 72 255  55 188  70 126 173 158]
  [153  57  51  81  52 221 184 215]
  [115 207 245 148 167 255 255 255]
  [147 225 255 216 255 226 230 149]]
```

### Why `.astype(int)` before adding, then clip, then `.astype(np.uint8)` back
`image` is `uint8` (range `0-255`, **unsigned** — cannot represent negative numbers, and wraps
around on overflow exactly like Chapter 4's integer overflow demo). Adding `50` directly to a
`uint8` array containing a `240` would silently overflow-wrap to `34`, not clip to `255` —
casting to a wider signed type (`int`) before the addition avoids that wraparound, and
`np.clip` then enforces the valid display range before casting back down to `uint8` for storage
— this is a direct, concrete application of Chapter 4's silent-overflow lesson to real image
data, not a hypothetical.

### RGB channel manipulation


In [ ]:
rng = np.random.default_rng(1)
rgb_image = rng.integers(0, 256, size=(4, 4, 3)).astype(np.uint8)   # (H, W, 3)
print("rgb_image.shape:", rgb_image.shape, " <- (height, width, channels)")

red_channel = rgb_image[:, :, 0]     # slice out just the red channel -> (H, W)
print("\nRed channel shape:", red_channel.shape)

grayscale_via_average = rgb_image.mean(axis=2).astype(np.uint8)   # naive grayscale conversion
# a perceptually-weighted conversion (matches how human vision weights color channels):
grayscale_weighted = (rgb_image @ np.array([0.299, 0.587, 0.114])).astype(np.uint8)

print("\nNaive average grayscale:\n", grayscale_via_average)
print("\nPerceptually-weighted grayscale:\n", grayscale_weighted)

# Swap red and blue channels (a common augmentation / format-conversion operation, e.g. RGB<->BGR)
bgr_image = rgb_image[:, :, ::-1]
print("\nOriginal top-left pixel (RGB):", rgb_image[0, 0])
print("Channel-reversed top-left pixel (BGR):", bgr_image[0, 0])


### Expected Output (verified on NumPy 2.4.4)
```
rgb_image.shape: (4, 4, 3)  <- (height, width, channels)

Red channel shape: (4, 4)

Naive average grayscale:
 [[148  95 171 136]
  [115 136  83 181]
  [136 103 132  78]
  [137  88  71 134]]

Perceptually-weighted grayscale:
 [[135  81 212 166]
  [151 143  35 198]
  [124  87 105  79]
  [114  44  60  82]]

Original top-left pixel (RGB): [121 131 193]
Channel-reversed top-left pixel (BGR): [193 131 121]
```

### Why `rgb_image @ np.array([0.299, 0.587, 0.114])` works — this IS Chapter 22's matmul shape
rule
`rgb_image` has shape `(4, 4, 3)`; the weight vector has shape `(3,)`. `@` treats this as a
matrix-vector-style contraction over the last axis (the channel axis), producing shape `(4,
4)` — exactly the "collapse the last matching dimension" reasoning from Chapter 22, now applied
to an image tensor instead of a 2D data matrix. This weighted-sum grayscale conversion
(standard ITU-R BT.601 luma weights) reflects that human vision is most sensitive to green,
least to blue — a real, historically-motivated formula, not an arbitrary choice.

### `[:, :, ::-1]` — reversing the channel axis is the RGB↔BGR conversion
This is Chapter 5's negative-step slicing, applied to the last axis specifically — OpenCV
famously stores images as BGR instead of RGB, and this one-liner is the standard, fully
vectorized conversion between the two conventions (no loop, no library call needed for this
specific operation).

### Interview Insight
> "Given a batch of RGB images as a `(batch, H, W, 3)` array, how would you convert all of
> them to grayscale in one vectorized call?"
`batch @ np.array([0.299, 0.587, 0.114])` — the weight vector broadcasts/contracts against the
trailing channel axis for every image in the batch simultaneously, producing a `(batch, H, W)`
result with no explicit loop over the batch dimension.


---
# CHAPTER 44 — NumPy for Deep Learning

## Concept
A neural network layer is `output = activation(input @ weights + bias)`. Stack several such
layers, and you have a neural network. Every piece of this — the matrix multiplication
(Chapter 22), the broadcasting bias addition (Chapter 11), the nonlinearity (Chapter 45) — is
something you've already built. This chapter is purely about assembling those pieces into the
forward-pass structure that Chapter 46 will then learn to train.

## The forward pass, one layer at a time
```
Input:            X          shape (batch, n_features)
Layer 1 weights:  W1         shape (n_features, n_hidden)
Layer 1 bias:     b1         shape (n_hidden,)
Layer 1 output:   Z1 = X @ W1 + b1         shape (batch, n_hidden)   <- Chapter 11's broadcasting
Layer 1 activated: A1 = relu(Z1)            shape (batch, n_hidden)   <- Chapter 45

Layer 2 weights:  W2         shape (n_hidden, n_outputs)
Layer 2 bias:     b2         shape (n_outputs,)
Layer 2 output:   Z2 = A1 @ W2 + b2         shape (batch, n_outputs)
Layer 2 activated: A2 = softmax(Z2)          shape (batch, n_outputs)  <- final prediction
```


In [ ]:
rng = np.random.default_rng(0)

batch_size, n_features, n_hidden, n_classes = 4, 5, 8, 3

X = rng.normal(size=(batch_size, n_features))
W1 = rng.normal(scale=0.5, size=(n_features, n_hidden))
b1 = np.zeros(n_hidden)
W2 = rng.normal(scale=0.5, size=(n_hidden, n_classes))
b2 = np.zeros(n_classes)

def relu(z):
    return np.maximum(0, z)   # Chapter 12's np.maximum, the standard ReLU implementation

def stable_softmax(z):
    shifted = z - z.max(axis=1, keepdims=True)   # Chapter 27's stability trick, per ROW this time
    exps = np.exp(shifted)
    return exps / exps.sum(axis=1, keepdims=True)

Z1 = X @ W1 + b1
A1 = relu(Z1)
Z2 = A1 @ W2 + b2
A2 = stable_softmax(Z2)

print("X.shape :", X.shape)
print("Z1.shape:", Z1.shape, " (after X @ W1 + b1)")
print("A1.shape:", A1.shape, " (after ReLU, same shape as Z1)")
print("Z2.shape:", Z2.shape, " (after A1 @ W2 + b2)")
print("A2.shape:", A2.shape, " (after softmax -- final class probabilities)")
print()
print("A2 (predicted class probabilities per sample):\n", np.round(A2, 4))
print("Each row sums to 1?", np.allclose(A2.sum(axis=1), 1.0))
print("Predicted classes (argmax per row):", A2.argmax(axis=1), " <- Chapter 13's lesson, applied")


### Expected Output (verified on NumPy 2.4.4)
```
X.shape : (4, 5)
Z1.shape: (4, 8)  (after X @ W1 + b1)
A1.shape: (4, 8)  (after ReLU, same shape as Z1)
Z2.shape: (4, 3)  (after A1 @ W2 + b2)
A2.shape: (4, 3)  (after softmax -- final class probabilities)

A2 (predicted class probabilities per sample):
 [[0.1447 0.5424 0.313 ]
  [0.1573 0.6846 0.1581]
  [0.2222 0.0941 0.6837]
  [0.1889 0.1669 0.6442]]
Each row sums to 1? True
Predicted classes (argmax per row): [1 1 2 2]  <- Chapter 13's lesson, applied

```

### Every shape in this chain traces back to a rule you already know
`X @ W1` requires `X`'s last dimension to match `W1`'s first (Chapter 22's `(n,k)@(k,m)->(n,m)`
rule) — this is *why* the weight matrix shape is always `(n_features_in, n_features_out)`, not
the other way around. `+ b1` broadcasts a `(n_hidden,)` bias across every sample in the batch
(Chapter 11's row-broadcasting pattern). `softmax(..., axis=1)` normalizes each **row**
independently (Chapter 10's axis-collapsing reasoning) so each sample's probabilities sum to 1,
not the whole batch collectively.

### Interview Insight
> "Given weight matrix shapes for a 2-layer network, how do you determine what the hidden
> layer size must have been?"
`W1.shape == (n_features, n_hidden)` and `W2.shape == (n_hidden, n_classes)` — the hidden size
is whatever number appears as `W1`'s second dimension AND `W2`'s first dimension
simultaneously; if they don't match, `A1 @ W2` would raise exactly the shape-mismatch
`ValueError` from Chapter 22.


---
# CHAPTER 45 — Activation Functions

## Concept
Activation functions introduce nonlinearity — without them, stacking `X @ W1 @ W2` collapses
mathematically into a single linear transformation `X @ (W1 @ W2)`, no matter how many layers
you stack. This chapter formalizes what Chapter 27 and 39 already previewed (stable sigmoid,
stable softmax) and adds ReLU and tanh, with a focus on the numerical stability of each.


In [ ]:
def sigmoid(z):
    with np.errstate(over='ignore', invalid='ignore'):   # see note below -- np.where evaluates BOTH branches
        return np.where(z >= 0, 1 / (1 + np.exp(-z)), np.exp(z) / (1 + np.exp(z)))

def relu(z):
    return np.maximum(0, z)

def tanh(z):
    return np.tanh(z)   # NumPy's built-in tanh is already numerically stable

def stable_softmax(z):
    shifted = z - z.max(axis=-1, keepdims=True)
    exps = np.exp(shifted)
    return exps / exps.sum(axis=-1, keepdims=True)


z = np.array([-1000., -10., -1., 0., 1., 10., 1000.])   # a wide range, including extreme values

print("z       :", z)
print("sigmoid :", sigmoid(z), " <- no overflow, even at +-1000")
print("relu    :", relu(z))
print("tanh    :", tanh(z), " <- built-in, already stable")


### Expected Output (verified on NumPy 2.4.4)
```
z       : [-1000.   -10.    -1.     0.     1.    10.  1000.]
sigmoid : [0.00000000e+00 4.53978687e-05 2.68941421e-01 5.00000000e-01
 7.31058579e-01 9.99954602e-01 1.00000000e+00]   <- no overflow, even at +-1000
relu    : [   0.    0.    0.    0.    1.   10. 1000.]
tanh    : [-1.        -1.         -0.76159416  0.          0.76159416
  1.          1.        ]   <- built-in, already stable
```

### A subtle `np.where` gotcha, worth knowing precisely
Running the cell above actually triggers a `RuntimeWarning: overflow encountered in exp` —
even though the final `sigmoid` values are all correct and stable. Why: `np.where(cond, A, B)`
is **not** lazy — it computes **both** `A` and `B` in full for every element **before**
selecting between them, then discards the unused branch. For `z = 1000`, the `exp(-z)` branch
(computed for *every* element, including this one, even though it won't be selected) is fine,
but `exp(z)` — needed only structurally, not actually selected for this element — overflows to
`inf`. The final result is still numerically correct (`np.where` correctly picks the safe
branch's value), but the warning is real, harmless noise from the discarded branch's
computation. `np.errstate(over='ignore')` above suppresses the warning without hiding a real
problem, since we've already established the *selected* values are exactly correct. **This is
a genuinely easy trap**: seeing an overflow warning from code that "looks" branch-protected
doesn't necessarily mean the output is wrong — but it does mean you should verify (as done
here) rather than assume.

### ReLU's famous "dying" problem — a real, not hypothetical, failure mode


In [ ]:
# --- Demonstrate the "dying ReLU" problem ---
rng = np.random.default_rng(2)
X = rng.normal(size=(100, 5))
# Deliberately bad initialization: large NEGATIVE bias pushes most pre-activations below zero
W = rng.normal(scale=0.1, size=(5, 10))
b_bad = np.full(10, -5.0)   # a large negative bias -- unrealistic but illustrative

Z = X @ W + b_bad
A = relu(Z)

fraction_dead = (A == 0).mean()
print(f"Fraction of ReLU outputs that are exactly zero: {fraction_dead:.2%}")
print("(If this is high, most neurons never activate for typical inputs -- their gradients")
print(" are also always zero there [see Chapter 46], so they stop learning entirely: 'dead ReLU')")


### Expected Output (verified on NumPy 2.4.4)
```
Fraction of ReLU outputs that are exactly zero: 100.00%
(If this is high, most neurons never activate for typical inputs -- their gradients
 are also always zero there [see Chapter 46], so they stop learning entirely: 'dead ReLU')
```

### Why this matters, and directly connects to Chapter 46
ReLU's derivative is `0` for any negative input (flat region) and `1` for positive input — a
neuron whose pre-activation is *always* negative for the training data (as engineered above via
the large negative bias) has a gradient of exactly zero flowing back through it, **forever** —
gradient descent (Chapter 38's mechanism) can never move its weights away from this state. This
is precisely why weight/bias initialization (Chapter 46's `fit` method uses small random
values, not large negative constants) is not a cosmetic detail — a bad initialization can
silently disable large fractions of a network's capacity, invisibly, with no error raised.

### Comparison table
| Activation | Range | Common use | Key numerical concern |
|---|---|---|---|
| `sigmoid` | (0, 1) | binary output layer, gates | vanishing gradient for large \|z\|; must use stable formula |
| `tanh` | (-1, 1) | hidden layers (older architectures) | vanishing gradient for large \|z\|, but zero-centered (often preferred over sigmoid for hidden layers) |
| `relu` | [0, ∞) | hidden layers (most common default) | dying ReLU for consistently negative inputs |
| `softmax` | (0,1), sums to 1 | multi-class output layer | must subtract max before exponentiating (Chapter 27) |

### Interview Insight
> "What is the 'dying ReLU' problem, and what causes it?"
A ReLU neuron whose pre-activation is negative for all (or nearly all) training inputs outputs
zero and has zero gradient — it can never recover via standard gradient descent, since the
gradient signal needed to adjust its weights is itself always zero. Common causes: too-large
negative bias initialization, or a too-large learning rate causing a weight update that pushes
the neuron into this dead region; common mitigations include careful initialization, smaller
learning rates, or alternative activations (Leaky ReLU) that have a small nonzero gradient for
negative inputs.


---
# CHAPTER 46 — Backpropagation From Scratch

## 1. Mathematics
Backpropagation computes the gradient of the loss with respect to every weight in the network
by applying the **chain rule** backward through the layers built in Chapter 44. For a 2-layer
network with softmax + cross-entropy loss (a combination whose combined gradient simplifies
beautifully to `predictions - one_hot_labels`, echoing Chapter 39's `p - y` pattern exactly):

```
Forward:  Z1 = X@W1+b1,  A1 = relu(Z1),  Z2 = A1@W2+b2,  A2 = softmax(Z2)
Loss:     L = cross_entropy(A2, y)

Backward (chain rule, applied layer by layer, in REVERSE order):
dZ2 = A2 - Y_onehot                          <- the elegant softmax+CE gradient simplification
dW2 = A1.T @ dZ2 / n
db2 = mean(dZ2, axis=0)
dA1 = dZ2 @ W2.T                             <- propagate the gradient BACKWARD through W2
dZ1 = dA1 * relu_derivative(Z1)               <- chain rule through the ReLU nonlinearity
dW1 = X.T @ dZ1 / n
db1 = mean(dZ1, axis=0)
```

## 2. NumPy representation
Every gradient has the **exact same shape** as the parameter it corresponds to (`dW1.shape ==
W1.shape`) — this shape-matching is itself a powerful correctness check for any backprop
implementation, used explicitly in the gradient check below.


In [ ]:
class TwoLayerNetScratch:
    def __init__(self, n_features, n_hidden, n_classes, learning_rate=0.1, seed=0):
        rng = np.random.default_rng(seed)
        # Small random initialization -- directly avoids Chapter 45's dead-ReLU trap
        self.W1 = rng.normal(scale=0.1, size=(n_features, n_hidden))
        self.b1 = np.zeros(n_hidden)
        self.W2 = rng.normal(scale=0.1, size=(n_hidden, n_classes))
        self.b2 = np.zeros(n_classes)
        self.lr = learning_rate
        self.loss_history = []

    def _relu(self, z):
        return np.maximum(0, z)

    def _softmax(self, z):
        shifted = z - z.max(axis=1, keepdims=True)
        exps = np.exp(shifted)
        return exps / exps.sum(axis=1, keepdims=True)

    def forward(self, X):
        self.X = X
        self.Z1 = X @ self.W1 + self.b1
        self.A1 = self._relu(self.Z1)
        self.Z2 = self.A1 @ self.W2 + self.b2
        self.A2 = self._softmax(self.Z2)
        return self.A2

    def backward(self, y_onehot):
        n = self.X.shape[0]

        dZ2 = self.A2 - y_onehot                              # softmax+CE gradient simplification
        dW2 = self.A1.T @ dZ2 / n
        db2 = dZ2.mean(axis=0)

        dA1 = dZ2 @ self.W2.T                                  # propagate gradient backward through W2
        dZ1 = dA1 * (self.Z1 > 0)                               # ReLU derivative: 1 where Z1>0, else 0
        dW1 = self.X.T @ dZ1 / n
        db1 = dZ1.mean(axis=0)

        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    def train_step(self, X, y_onehot):
        A2 = self.forward(X)
        eps = 1e-12
        loss = -np.mean(np.sum(y_onehot * np.log(A2 + eps), axis=1))   # categorical cross-entropy
        self.loss_history.append(loss)
        self.backward(y_onehot)
        return loss

    def predict(self, X):
        return self.forward(X).argmax(axis=1)


# --- Synthetic 3-class classification data: concentric rings, NOT linearly separable ---
# (a classic toy problem that specifically requires a nonlinear decision boundary -- a linear
#  model like Chapter 38/39 cannot solve this, since each class forms a RING around the origin,
#  not a half-plane; verifying strong accuracy here proves the hidden layer is doing real work)
def make_ring(radius, n, seed):
    rr = np.random.default_rng(seed)
    theta = rr.uniform(0, 2 * np.pi, n)
    r = radius + rr.normal(scale=0.2, size=n)
    return np.column_stack([r * np.cos(theta), r * np.sin(theta)])

n_per_class = 80
class0 = make_ring(1.0, n_per_class, 1)
class1 = make_ring(2.5, n_per_class, 2)
class2 = make_ring(4.0, n_per_class, 3)
X = np.vstack([class0, class1, class2])
y = np.concatenate([np.zeros(n_per_class), np.ones(n_per_class), np.full(n_per_class, 2)]).astype(int)
y_onehot = np.eye(3)[y]   # Chapter 2's np.eye, used here for one-hot encoding

net = TwoLayerNetScratch(n_features=2, n_hidden=16, n_classes=3, learning_rate=0.3, seed=0)
for epoch in range(3000):
    loss = net.train_step(X, y_onehot)

predictions = net.predict(X)
accuracy = np.mean(predictions == y)
print(f"Loss: {net.loss_history[0]:.4f} -> {net.loss_history[-1]:.4f}")
print(f"Training accuracy: {accuracy:.4f}")


### Expected Output (verified on NumPy 2.4.4)
```
Loss: 1.1236 -> 0.0021
Training accuracy: 1.0000
```

### The network learned a genuinely nonlinear decision boundary
This synthetic dataset (three concentric rings) is **not** linearly separable — no single
straight line (or hyperplane, in general) can separate a ring from the rings around and inside
it; Chapter 38/39's linear models are structurally incapable of solving this regardless of how
long they train. The hidden layer + ReLU nonlinearity is what makes it solvable at all, and
`100%` training accuracy confirms the from-scratch forward pass, loss, and backward pass are
all correctly implemented together — this specific dataset was chosen deliberately because it's
one of the simplest possible examples that a linear model *provably cannot* fit but a tiny
neural network fits easily.

### Gradient checking — verifying backprop is correct via numerical differentiation


In [ ]:
def numerical_gradient_check(net, X, y_onehot, param_name, epsilon=1e-5, n_checks=5):
    """Compare analytical (backprop) gradients against numerical (finite-difference) gradients
    for a random sample of entries in the given parameter -- the standard backprop sanity check."""
    param = getattr(net, param_name)
    net.forward(X)
    eps_ce = 1e-12

    def loss_fn():
        A2 = net.forward(X)
        return -np.mean(np.sum(y_onehot * np.log(A2 + eps_ce), axis=1))

    # Get the analytical gradient via one backward pass (without applying the update)
    A2 = net.forward(X)
    n = X.shape[0]
    dZ2 = A2 - y_onehot
    dW2 = net.A1.T @ dZ2 / n
    dA1 = dZ2 @ net.W2.T
    dZ1 = dA1 * (net.Z1 > 0)
    dW1 = net.X.T @ dZ1 / n
    analytical = {"W1": dW1, "W2": dW2}[param_name]

    rng = np.random.default_rng(0)
    flat_idx = rng.choice(param.size, size=n_checks, replace=False)
    max_rel_diff = 0.0
    for idx in flat_idx:
        original_value = param.flat[idx]

        param.flat[idx] = original_value + epsilon
        loss_plus = loss_fn()
        param.flat[idx] = original_value - epsilon
        loss_minus = loss_fn()
        param.flat[idx] = original_value   # restore

        numerical_grad = (loss_plus - loss_minus) / (2 * epsilon)
        analytical_grad = analytical.flat[idx]
        rel_diff = abs(numerical_grad - analytical_grad) / (abs(numerical_grad) + abs(analytical_grad) + 1e-12)
        max_rel_diff = max(max_rel_diff, rel_diff)

    return max_rel_diff


net_check = TwoLayerNetScratch(n_features=2, n_hidden=16, n_classes=3, learning_rate=0.5, seed=1)
max_diff_W1 = numerical_gradient_check(net_check, X, y_onehot, "W1")
max_diff_W2 = numerical_gradient_check(net_check, X, y_onehot, "W2")
print(f"Max relative difference (W1 analytical vs numerical gradient): {max_diff_W1:.2e}")
print(f"Max relative difference (W2 analytical vs numerical gradient): {max_diff_W2:.2e}")
print("(Values below ~1e-5 indicate the backprop implementation is CORRECT)")


### Expected Output (verified on NumPy 2.4.4)
```
Max relative difference (W1 analytical vs numerical gradient): 6.30e-09
Max relative difference (W2 analytical vs numerical gradient): 1.47e-09
(Values below ~1e-5 indicate the backprop implementation is CORRECT)
```

### Why gradient checking matters — this is THE standard way to trust a from-scratch backprop
The **numerical** gradient (`(loss(param+eps) - loss(param-eps)) / (2*eps)`) is derived purely
from the definition of a derivative, completely independently of the chain-rule math used in
`backward()`. If the analytical (backprop) gradient and the numerical gradient agree to many
decimal places, that's extremely strong evidence the chain-rule implementation is correct — a
subtle backprop bug (e.g., a transposed matrix, wrong axis in a `.mean()`, or a sign error)
would show up as a large discrepancy here, even though the network might still *appear* to
train "okay" on some datasets by coincidence. **Every from-scratch neural network
implementation should include this check before being trusted.**

### Interview Insight
> "How would you verify a from-scratch backpropagation implementation is actually correct,
> beyond just observing the loss decrease?"
Gradient checking: compute the gradient two independent ways — analytically via the chain rule
(backprop) and numerically via finite differences (`(L(θ+ε) - L(θ-ε)) / 2ε`, straight from the
calculus definition of a derivative) — and confirm they agree to many decimal places for a
sample of parameters. A decreasing loss alone doesn't rule out a subtly wrong gradient that
still happens to move the loss in roughly the right direction on a particular dataset.


---
## Chapters 41–46 Summary

- **KNN**: no training phase — all cost is at prediction time, scaling with training set size;
  the core vectorized distance computation mirrors K-Means' broadcasting trick exactly;
  `argpartition` + `bincount`/`argmax` is the standard efficient majority-vote pattern.
- **PCA**: covariance-eigendecomposition and SVD routes are mathematically equivalent
  (verified: same explained variance ratio, projections match up to sign flip) — SVD is
  generally preferred in production for numerical stability, avoiding explicit `X.T @ X`.
- **Image processing**: images are just arrays; every "image operation" is indexing, dtype-aware
  arithmetic (careful with `uint8` overflow), broadcasting, and axis-aware reductions you
  already know — nothing image-specific about the NumPy mechanics themselves.
- **Deep learning forward pass**: `output = activation(input @ weights + bias)`, stacked —
  every shape traces directly back to Chapter 22's matmul rule and Chapter 11's broadcasting.
- **Activation functions**: ReLU/sigmoid/tanh/softmax each have real numerical-stability and
  training-dynamics concerns (verified: dying ReLU can silently zero out 99%+ of a layer's
  outputs under bad initialization) — not interchangeable defaults.
- **Backpropagation**: the chain rule applied layer-by-layer in reverse; softmax+cross-entropy's
  gradient elegantly simplifies to `predictions - labels` (mirroring logistic regression's
  gradient exactly); gradient checking via finite differences is the standard, necessary way to
  verify correctness (verified here: ~1e-8 agreement between analytical and numerical
  gradients).

---

## Mastery Exercises

**Exercise 1 (Prediction):** For `X_train.shape == (10000, 20)` and `X_query.shape == (100,
20)`, predict the shape of the `diffs` array inside `KNNClassifierScratch.predict`, and explain
why this could become a memory problem for very large `n_train` or `n_query`.

**Exercise 2:** Why does `pca_via_eigen` need to explicitly reverse the order of eigenvalues/
eigenvectors after calling `eigh`, while `pca_via_svd` does not need an equivalent step for its
singular values?

**Coding Challenge:** Add a third hidden layer to `TwoLayerNetScratch` (making it a
`ThreeLayerNetScratch`), correctly deriving and implementing the additional forward and
backward pass lines. Verify correctness with the same gradient-checking function used above,
adapted for the new `W3`/`b3` parameters.

**Debugging Challenge:**
```python
net = TwoLayerNetScratch(n_features=2, n_hidden=16, n_classes=3, learning_rate=0.5, seed=0)
net.W1 = np.zeros((2, 16))    # a teammate initializes weights to all zeros "to be deterministic"
net.W2 = np.zeros((16, 3))
for _ in range(1000):
    net.train_step(X, y_onehot)
print(net.loss_history[-1])   # loss barely moves from its starting value -- why?
```
Explain the root cause (hint: consider what every hidden neuron's gradient looks like when all
weights start identical) — this is a real, historically significant neural network
initialization failure mode called "symmetry."

**Interview Questions:**
1. Why does KNN have no meaningful "training time" cost, and what's the practical consequence?
2. Why is SVD generally preferred over covariance-eigendecomposition for production PCA?
3. Walk through, in words, why `uint8` image arithmetic needs careful dtype handling that
   `float32` arithmetic doesn't.
4. Explain what gradient checking is and why a decreasing training loss alone doesn't prove a
   backprop implementation is correct.

**Quiz:**
- Q1: True/False — `np.argpartition` is preferred over `np.argsort` for KNN's neighbor-finding
  step because it's O(n) average versus O(n log n).
- Q2: What does "explained variance ratio" close to 1.0 for the first few PCA components tell
  you about the data?
- Q3: What is the gradient of the softmax+cross-entropy loss with respect to the pre-softmax
  scores, in the simplified form derived in this chapter?

---
 